# Pokémon Battle Prediction — FDS Kaggle Challenge (Gen 2 OU)
### Team Random Forretress
**Team Members:** Cristian Fraschetti, Daniele Mastropietro, Andrea Mancini, Ariana Sadyr  
**Sapienza University of Rome — Fundamentals of Data Science**

---
## 1. Project Overview & Competition Guidelines

The goal of this competition is to build a supervised binary classification model capable of predicting the winner of a Pokémon battle in the competitive *Generation 2 OverUsed (OU)* format (`player_won`: `1` if Player 1 wins, `0` if Player 1 loses), observing **only the initial team compositions and the first 30 active turns**.

### Official Competition Constraints:
* **Allowed Data:** Strictly the provided competition files (`train.jsonl`, `test.jsonl`, and `type_chart_gen2.csv`). No external databases, Smogon usage stats, or external scrapers.
* **Classical ML Only:** Strictly classical Machine Learning algorithms (Logistic Regression, LightGBM, CatBoost, XGBoost, Ensembles). **Neural networks are prohibited**.
* **Submission Format:** 5,000 rows matching `sample_submission.csv` (columns `id` and `player_won`).
* **Official Submissions:** As documented in our paper, our **3 official Kaggle submissions** come from **Version 2 (Soft Voting)**, **Version 3 (Bagged Ensemble)**, and **Version 4 (Stacking Ensemble)**.


---
## 2. Environment Setup & Repository Cloning (Course Requirement)

Per i requisiti delle slide dell'esame di *Fundamentals of Data Science*, il notebook deve essere autosufficiente su Google Colab o su qualsiasi ambiente remoto. La cella seguente clona automaticamente il repository ufficiale del team da GitHub e installa le dipendenze necessarie se eseguito in un ambiente cloud.


In [ ]:
import os
import sys
from pathlib import Path

# Configurazione repository GitHub ufficiale (Requisito slide FDS)
REPO_URL = "https://github.com/FrascaHoryzont/Pokemon-Battle-Prediction.git"
BRANCH = "clearcode"
REPO_NAME = "Pokemon-Battle-Prediction"

# Se siamo su Google Colab o ambiente senza i dati locali, cloniamo la repo
if not Path("data").exists() and not Path("type_chart_gen2.csv").exists():
    print(f"Ambiente remoto rilevato. Clonazione da {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
    %cd {REPO_NAME}
    !pip install -r requirements.txt
    print("Repository clonata e dipendenze installate con successo!")
else:
    print(f"Ambiente locale rilevato. Directory di lavoro: {Path.cwd()}")

# Assicuriamo che la radice del progetto e src siano nel path di Python
REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)
print("Ambiente Python e librerie configurate con successo. Random Seed = 42.")


---
## 3. Exploratory Data Analysis (EDA) & Domain Discoveries

Tutte le funzioni di analisi e visualizzazione sono definite **direttamente in questo notebook** per garantire la massima trasparenza e riproducibilità.

Analizziamo le due anomalie empiriche fondamentali evidenziate nella Sezione 3 della nostra relazione:
1. **Il paradosso dei KO non registrati di Player 2:** Player 1 registra in media 1.69 faint al turno 30 contro solo 0.50 di Player 2. L'euristica che considera fainted un Pokémon che lascia il campo con $\le 20\%$ HP e non rientra più porta la regola *"chi ha meno fainted vince"* dal **58.08% al 64.47%**.
2. **L'impatto decisivo delle Spikes:** Player 1 vince il **60.2%** delle partite con le proprie Spikes attive, contro il **39.7%** se sono attive solo quelle di Player 2.


In [ ]:
# ==============================================================================
# Funzioni grafiche definite direttamente nel notebook (self-contained)
# ==============================================================================

def plot_ko_asymmetry():
    categories = ["P1 Registrati", "P2 Grezzi (Raw)", "P2 con Euristica (<=20% HP)"]
    values = [1.69, 0.50, 1.62]
    colors = ["#2b5c8f", "#d95f02", "#2ca02c"]

    fig, ax = plt.subplots(figsize=(8, 4), dpi=100)
    bars = ax.bar(categories, values, color=colors, width=0.5, edgecolor="black", linewidth=0.8)
    ax.set_ylabel("Media Pokemon Fainted al Turno 30", fontsize=11)
    ax.set_title("Asimmetria dei KO di Player 2 e Ricostruzione Euristica", fontsize=12, pad=12, fontweight="bold")
    ax.set_ylim(0, 2.2)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 4), textcoords="offset points", ha="center", va="bottom",
                    fontweight="bold", fontsize=10)
    plt.tight_layout()
    plt.show()

def plot_spikes_impact():
    labels = ["Solo Spikes P1", "Nessuna Spikes", "Entrambe Attive", "Solo Spikes P2"]
    win_rates = [60.2, 50.1, 50.8, 39.7]
    colors = ["#1f77b4" if wr >= 50 else "#d62728" for wr in win_rates]

    fig, ax = plt.subplots(figsize=(7.5, 4), dpi=100)
    bars = ax.bar(labels, win_rates, color=colors, width=0.5, edgecolor="black", linewidth=0.8)
    ax.axhline(50.0, color="black", linestyle="--", linewidth=1.0, label="Baseline Neutra (50%)")
    ax.set_ylabel("Win Rate P1 (%)", fontsize=11)
    ax.set_title("Impatto delle Spikes sull'Esito della Battaglia (Turno 30)", fontsize=12, pad=12, fontweight="bold")
    ax.set_ylim(30, 70)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.legend(loc="upper right")

    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 4), textcoords="offset points", ha="center", va="bottom",
                    fontweight="bold", fontsize=10)
    plt.tight_layout()
    plt.show()

plot_ko_asymmetry()
plot_spikes_impact()


---
## 4. Strategia di Validazione & Prevenzione Rigorosa del Data Leakage

Come descritto nella Sezione 2 della relazione:
* Utilizziamo una **Repeated Stratified 5-Fold Cross-Validation (15 fold totali, 5 fold x 3 shuffle con seed 42)** per minimizzare il rumore statistico intrinseco di LightGBM ($\pm 0.4\%$).
* **Nessun Data Leakage:** Tutti i passaggi dipendenti dai dati (ranking delle feature per noise pruning, target encoding, KO detector) vengono eseguiti **rigorosamente all'interno del fold di training**. Selezionare le feature sull'intero dataset prima della cross-validazione gonfiava artificialmente la stima da **68.51% a 69.21%**.


In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
print(f"Setup di validazione: 5 fold x 3 ripetizioni ({cv.get_n_splits()} split totali).")


---
## 5. Addestramento dei Modelli & Generazione delle 3 Submission di Kaggle

Come specificato nella relazione, le nostre **3 submission ufficiali per la competizione Kaggle** provengono dalla **Second Version (V2)**, **Third Version (V3)** e **Fourth Version (V4)**.

Per garantire la massima fedeltà scientifica, **non modifichiamo una singola riga di codice né gli iperparametri originali dei modelli**: eseguiamo direttamente gli script del repository (`second_version/train.py`, `third_version/train.py`, `fourth_version/train.py`), che generano ciascuno il proprio file di submission ufficiale e ne stampano l'accuratezza in cross-validazione.


### 5.1 Kaggle Submission 1: Version 2 (Soft Voting LightGBM + CatBoost)
* **Architettura:** Soft Voting tra LightGBM (alberi a 4 foglie, tuned via random search su log-loss) e CatBoost (alberi simmetrici a profondità 6).
* **Feature:** 470 feature (duello attivi, HP per specie, survivor matchup, lookup roster P2).
* **Script originale:** `second_version/train.py`
* **Output generato:** `second_version/submission.csv`


In [ ]:
# Esecuzione script originale Version 2
!python second_version/train.py

# Verifica della submission generata
sub_v2 = pd.read_csv("second_version/submission.csv")
print(f"\n[Kaggle Submission 1 - V2 Soft Voting]")
print(f"File: second_version/submission.csv | Dimensioni: {sub_v2.shape}")
print(f"P1 Win Rate predetta: {sub_v2['player_won'].mean():.2%}")
assert len(sub_v2) == 5000, "La submission deve contenere esattamente 5.000 righe!"
sub_v2.head()


### 5.2 Kaggle Submission 2: Version 3 (Tri-Model Bagged Ensemble XGB / LGB / CAT)
* **Architettura:** Tri-Model Bagged Blend (60% XGBoost, 30% LightGBM, 10% CatBoost) con soglia calibrata ($\tau = 0.505$) per compensare l'asimmetria di visibilità P1/P2.
* **Feature:** Noise pruning con XGBoost gain che seleziona le migliori 200 feature da 577 colonne di dominio Gen 2 OU.
* **Script originale:** `third_version/train.py`
* **Output generato:** `third_version/submission.csv`


In [ ]:
# Esecuzione script originale Version 3
!python third_version/train.py

# Verifica della submission generata
sub_v3 = pd.read_csv("third_version/submission.csv")
print(f"\n[Kaggle Submission 2 - V3 Bagged Blend]")
print(f"File: third_version/submission.csv | Dimensioni: {sub_v3.shape}")
print(f"P1 Win Rate predetta: {sub_v3['player_won'].mean():.2%}")
assert len(sub_v3) == 5000, "La submission deve contenere esattamente 5.000 righe!"
sub_v3.head()


### 5.3 Kaggle Submission 3: Version 4 (Two-Level Stacking Ensemble con Meta-Learner)
* **Architettura:** Stacking a due livelli:
  * **Level 0:** Bagging di 15 fold con XGBoost, LightGBM e CatBoost sulle top 200 feature.
  * **Level 1 (Meta-Learner):** Logistic Regression regolarizzata $L_2$ addestrata sulle probabilità Out-Of-Fold, che impara i pesi ottimali (~35% XGB, ~33% LGB, ~32% CAT) e calibra la soglia decisionale in automatico tramite l'intercetta.
* **Script originale:** `fourth_version/train.py`
* **Output generato:** `fourth_version/submission.csv` (e `submission.csv` nella root del progetto)


In [ ]:
# Esecuzione script originale Version 4
!python fourth_version/train.py

# Verifica della submission generata
sub_v4 = pd.read_csv("fourth_version/submission.csv")
print(f"\n[Kaggle Submission 3 - V4 Two-Level Stacking]")
print(f"File: fourth_version/submission.csv | Dimensioni: {sub_v4.shape}")
print(f"P1 Win Rate predetta: {sub_v4['player_won'].mean():.2%}")
assert len(sub_v4) == 5000, "La submission deve contenere esattamente 5.000 righe!"
sub_v4.head()


---
## 6. Confronto e Verifica di Integrità delle 3 Submission di Kaggle

Confrontiamo le 3 submission ufficiali generate:
1. Controllo di allineamento degli ID con `sample_submission.csv` (ID da 0 a 4999).
2. Valutazione della concordanza tra i 3 modelli sui 5.000 test sample.
3. Distribuzione delle predizioni.


In [ ]:
# Caricamento del template ufficiale di submission
sample = pd.read_csv("sample_submission.csv")

# Verifica formale per ciascuna submission
submissions = {
    "V2 (Soft Voting)": sub_v2,
    "V3 (Bagged Ensemble)": sub_v3,
    "V4 (Stacking Ensemble)": sub_v4,
}

for name, sub in submissions.items():
    assert len(sub) == 5000, f"{name}: attese 5.000 righe, trovate {len(sub)}"
    assert list(sub.columns) == ["battle_id", "player_won"] or list(sub.columns) == ["id", "player_won"], f"{name}: colonne non conformi"
    assert set(sub["player_won"].unique()).issubset({0, 1}), f"{name}: predizioni non binarie"
    print(f"✓ {name}: Integrità verificata con successo (5.000 righe binarie).")

# Matrice di concordanza tra le 3 submission
agreement_v2_v3 = (sub_v2["player_won"] == sub_v3["player_won"]).mean()
agreement_v2_v4 = (sub_v2["player_won"] == sub_v4["player_won"]).mean()
agreement_v3_v4 = (sub_v3["player_won"] == sub_v4["player_won"]).mean()

print(f"\n--- Concordanza tra le predizioni di Test ---")
print(f"Concordanza V2 vs V3: {agreement_v2_v3 * 100:.2f}%")
print(f"Concordanza V2 vs V4: {agreement_v2_v4 * 100:.2f}%")
print(f"Concordanza V3 vs V4: {agreement_v3_v4 * 100:.2f}%")

# Salva copie esplicite per upload su Kaggle
sub_v2.to_csv("submission_v2_voting.csv", index=False)
sub_v3.to_csv("submission_v3_bagged.csv", index=False)
sub_v4.to_csv("submission_v4_stacking.csv", index=False)
print("\nFile di submission pronti per il download o submit Kaggle:")
print(" - submission_v2_voting.csv")
print(" - submission_v3_bagged.csv")
print(" - submission_v4_stacking.csv")


---
## 7. Tabella Riassuntiva Finale & Conclusioni della Relazione

Riproduciamo la tabella di sintesi della **Sezione 5 della relazione LaTeX**:

$$\begin{array}{lllr}
\hline
\textbf{Kaggle Submission} & \textbf{Versione} & \textbf{Modello} & \textbf{CV Accuracy} \\
\hline
\text{Baseline (Riferimento)} & \text{First} & \text{Logistic Regression, 71 features} & 66.88\% \\
\textbf{Submission 1} & \textbf{Second} & \textbf{LightGBM + CatBoost Soft Voting, 470 features} & \textbf{68.64\%} \\
\textbf{Submission 2} & \textbf{Third} & \textbf{XGB, LGBM, Cat fixed blend, 200 features} & \textbf{68.87\% / 68.54\%} \\
\textbf{Submission 3} & \textbf{Fourth} & \textbf{Two-Level Stacking Ensemble (Meta-LR), 200 features} & \textbf{69.11\% / 68.55\%} \\
\hline
\end{array}$$

### Conclusioni Metodologiche del Team:
1. **Pareggio statistico tra le 3 submission:** Come verificato tramite il test con 2.000 ricampionamenti di bootstrap nel paper, i modelli V2, V3 e V4 si attestano tutti nell'intervallo **68.5% – 69.1%**, confermando che la leaderboard pubblica (~68%) rispecchia esattamente le nostre stime out-of-fold.
2. **Il limite informativo del turno 30:** Un terzo delle battaglie al turno 30 è un testa o croce bilanciato ($0.45 \le \hat{p} \le 0.55$), dove l'accuratezza è solo del 56% e in cui si concentra il 76% di tutti gli errori di classificazione: le battaglie più combattute vengono decise nei turni successivi (turni 31–80).
3. **Metodologia contro il Leakage:** La lezione più importante del progetto è stata metodologica: comprendere le peculiarità del dato (i KO nascosti di P2), adottare una metrica rigorosa a 15 fold, e mantenere ogni singolo passaggio data-driven all'interno della cross-validazione.


In [ ]:
summary_table = pd.DataFrame({
    "Submission": ["—", "Kaggle Sub 1", "Kaggle Sub 2", "Kaggle Sub 3"],
    "Versione": ["First Version (V1)", "Second Version (V2)", "Third Version (V3)", "Fourth Version (V4)"],
    "Architettura": [
        "Logistic Regression (71 feat)",
        "Soft Voting: LightGBM + CatBoost (470 feat)",
        "Tri-Model Bagged Blend (XGB 60%, LGB 30%, CAT 10%, 200 feat)",
        "Two-Level Stacking Ensemble (XGB+LGB+CAT -> Meta LR, 200 feat)",
    ],
    "CV Accuracy (15-fold)": ["66.88% ± 0.61%", "68.64% ± 0.86%", "68.87% ± 0.77%", "69.11% ± 0.78%"],
    "Output File": [
        "first_version/submission.csv",
        "second_version/submission.csv",
        "third_version/submission.csv",
        "fourth_version/submission.csv",
    ]
})

print("==========================================================================================")
print("                   RIASSUNTO DELLE SUBMISSION E BENCHMARK FINALI                          ")
print("==========================================================================================")
display(summary_table) if 'display' in globals() else print(summary_table.to_string(index=False))
